# Temporal Graphs in pathpyG

*August 4 2026*  
*Training Workshop: Causality-Aware Temporal Networks*  
*Ingo Scholtes, CAIDAS, Julius-Maximilians-Universität Würzburg (JMU), Germany*  

In this notebook we will introduce the representation of temporal graph data using the `TemporalGraph` class and how such data can be used to calculate shortest time respecting paths between nodes as well temporal node cemtralities.

In [3]:
import pandas as pd
import torch
from torch_geometric.data import Data

import pathpyG as pp

We can create a temporal graph object from a list of time-stamped edges. Since `TemporalGraph` is a subclass of the `Graph` class, the internal structures are very similar:

In [4]:
tedges = [('a', 'b', 1),('a', 'b', 2), ('b', 'a', 3), ('b', 'c', 3), ('d', 'c', 4), ('a', 'b', 4), ('c', 'b', 4),
              ('c', 'd', 5), ('b', 'a', 5), ('c', 'b', 6)]
t = pp.TemporalGraph.from_edge_list(tedges)
print(t.mapping)
print(t.n)
print(t.m)

a -> 0
b -> 1
c -> 2
d -> 3

4
10


By default, all temporal graphs are directed. We can create an undirected version a temporal graph as follows:

In [5]:
x = t.to_undirected()
print(x.mapping)
print(x.n)
print(x.m)

a -> 0
b -> 1
c -> 2
d -> 3

4
20


We can also directly create a temporal graph from an instance of `pyG.TemporalData`

In [6]:
td = Data(
    edge_index = torch.Tensor([[0,1,2,0],[1,2,3,1]]).long(),
    time = torch.Tensor([0,1,2,3])
)
print(td)
t2 = pp.TemporalGraph(td)
print(t2)

Data(edge_index=[2, 4], time=[4])
Temporal Graph with 4 nodes, 3 unique edges and 4 events in [0.0, 3.0]
{'Edge Attributes': {}, 'Graph Attributes': {}, 'Node Attributes': {}}


/opt/conda/lib/python3.11/site-packages/torch_geometric/data/storage.py:452: UserWarning: Unable to accurately infer 'num_nodes' from the attribute set '{'edge_index', 'time'}'. Please explicitly set 'num_nodes' as an attribute of 'data' to suppress this warning
  warnings.warn(


We can restrict a temporal graph to a time window, which returns a temporal graph that only contains time-stamped edges in the given time interval.

In [7]:
t1 = t.get_window(0,4)
print(t1)
print(t1.m)
print(t1.start_time)
print(t1.end_time)

Temporal Graph with 4 nodes, 5 unique edges and 7 events in [1, 4]
{'Edge Attributes': {}, 'Graph Attributes': {'num_nodes': "<class 'int'>"}, 'Node Attributes': {}}
7
1
4


We can also extract a TemporalGraph object for a batch of temporal edges, which is defined by the start and end index of the edges defining the batch.

In [8]:
t1 = t.get_batch(1,6)
print(t1)
print(t1.m)
print(t1.start_time)
print(t1.end_time)

Temporal Graph with 4 nodes, 4 unique edges and 5 events in [2, 4]
{'Edge Attributes': {}, 'Graph Attributes': {}, 'Node Attributes': {}}
5
2
4


We can easily convert a temporal graph into a weighted time-aggregated static graph, where edge weights count the number of occurrences of an edge across all timestamps.

In [9]:
g = t.to_static_graph(weighted=True)
print(g)

Directed graph with 4 nodes and 6 edges
{'Edge Attributes': {'edge_weight': "<class 'torch.Tensor'> -> torch.Size([6])"}, 'Graph Attributes': {'num_nodes': "<class 'int'>"}, 'Node Attributes': {}}


We can also aggregate a temporal graph within a certain time window:

In [10]:
g = t.to_static_graph(time_window=(1, 3), weighted=True)
print(g)

Directed graph with 2 nodes and 1 edges
{'Edge Attributes': {'edge_weight': "<class 'torch.Tensor'> -> torch.Size([1])"}, 'Graph Attributes': {'num_nodes': "<class 'int'>"}, 'Node Attributes': {}}


Finally, we can use the class `RollingTimeWindow` to perform a rolling window analysis. The class returns an iterable object, where each iteration yields a time-aggregated weighted graph object as well as the corresponding time window.

In [11]:
r = pp.algorithms.RollingTimeWindow(t, window_size=3, step_size=1, return_window=True)
for g, w in r:
    print('Time window ', w)
    print(g)
    print(g.data.edge_index)
    print('---')

Time window  (1, 4)
Directed graph with 3 nodes and 3 edges
{'Edge Attributes': {'edge_weight': "<class 'torch.Tensor'> -> torch.Size([3])"}, 'Graph Attributes': {'num_nodes': "<class 'int'>"}, 'Node Attributes': {}}
EdgeIndex([[0, 1, 1],
           [1, 0, 2]], sparse_size=(3, 3), nnz=3, sort_order=row)
---
Time window  (2, 5)
Directed graph with 4 nodes and 5 edges
{'Edge Attributes': {'edge_weight': "<class 'torch.Tensor'> -> torch.Size([5])"}, 'Graph Attributes': {'num_nodes': "<class 'int'>"}, 'Node Attributes': {}}
EdgeIndex([[0, 1, 1, 2, 3],
           [1, 0, 2, 1, 2]], sparse_size=(4, 4), nnz=5, sort_order=row)
---
Time window  (3, 6)
Directed graph with 4 nodes and 6 edges
{'Edge Attributes': {'edge_weight': "<class 'torch.Tensor'> -> torch.Size([6])"}, 'Graph Attributes': {'num_nodes': "<class 'int'>"}, 'Node Attributes': {}}
EdgeIndex([[0, 1, 1, 2, 2, 3],
           [1, 0, 2, 1, 3, 2]], sparse_size=(4, 4), nnz=6, sort_order=row)
---
Time window  (4, 7)
Directed graph with 4 n

We can visualize temporal graphs using the plot function just like static graphs:

In [26]:
pp.plot(t, node_label=t.nodes, edge_color='lightgray');

Besides the standard formatting options available in pathpyG, temporal plots come with specific options tailored to their unique nature. These specialized settings allow for precise control over the time dimension of the visualization. The delta option lets you adjust the progression speed through the time steps of your visualization. Here, a value of 1000 translates to a one-second interval, providing a way to calibrate the pace at which the temporal data unfolds.

In [ ]:
color = {"a": "blue", "b": "red", "c": "green", "d": "yellow"}
pp.plot(t, node_color=color, delta=2500);

The source nodes, destination nodes and timestamps of time-stamped edges are stored as a `pyG TemporalData` object, which we can access in the following way.

In [13]:
t.data

Data(edge_index=[2, 10], time=[10], num_nodes=4)

In [14]:
print(t.data.edge_index)

EdgeIndex([[0, 0, 1, 1, 3, 0, 2, 2, 1, 2],
           [1, 1, 0, 2, 2, 1, 1, 3, 0, 1]], sparse_size=(4, 4), nnz=10)


In [15]:
print(t.data.time)

tensor([1, 2, 3, 3, 4, 4, 4, 5, 5, 6])


With the generator functions `edges` and `temporal_edges` we can iterate through the time-ordered (temporal) multi-edges of a temporal graph.

In [16]:
for v, w in t.edges:
    print(v, w)

a b
a b
b a
b c
d c
a b
c b
c d
b a
c b


In [17]:
for v, w, time in t.temporal_edges:
    print(v, w, time)

a b 1
a b 2
b a 3
b c 3
d c 4
a b 4
c b 4
c d 5
b a 5
c b 6


## Reading and writing temporal graph data

In [21]:
tedges = [('a', 'b', 1),('a', 'b', 2), ('b', 'a', 3), ('b', 'c', 3), ('d', 'c', 4), ('a', 'b', 4), ('c', 'b', 4),
              ('c', 'd', 5), ('b', 'a', 5), ('c', 'b', 6)]
t = pp.TemporalGraph.from_edge_list(tedges)
df = pp.io.temporal_graph_to_df(t)
print(df)

   v  w  t
0  a  b  1
1  a  b  2
2  b  a  3
3  b  c  3
4  d  c  4
5  a  b  4
6  c  b  4
7  c  d  5
8  b  a  5
9  c  b  6


In [22]:
t = pp.io.df_to_temporal_graph(df)
print(t)

Temporal Graph with 4 nodes, 6 unique edges and 10 events in [1, 6]
{'Edge Attributes': {}, 'Graph Attributes': {'num_nodes': "<class 'int'>"}, 'Node Attributes': {}}


In [23]:
df = pd.DataFrame([['a', 'b', 1], ['b', 'c', 2], ['a', 'c', 3]])
print(df)
t = pp.io.df_to_temporal_graph(df)
print(t)

   0  1  2
0  a  b  1
1  b  c  2
2  a  c  3
Temporal Graph with 3 nodes, 3 unique edges and 3 events in [1, 3]
{'Edge Attributes': {}, 'Graph Attributes': {'num_nodes': "<class 'int'>"}, 'Node Attributes': {}}


In [ ]:
pp.io.write_csv(t, path_or_buf='data/test_temporal_graph.csv')

In [ ]:
t = pp.io.read_csv_temporal_graph('data/test_temporal_graph.csv')
print(t)